In [ ]:
# Importer les packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Description des variables

Le jeu de données contient **1 000 trajets de taxi** décrits par **11 variables** : 10 variables explicatives et 1 variable cible (`Trip_Price`).

| Variable | Type | Description | Valeurs / plage observée |
|---|---|---|---|
| `Trip_Distance_km` | Numérique (continue) | Distance parcourue pendant le trajet, en kilomètres | 1,23 – 146,07 km (médiane ≈ 25,8) |
| `Time_of_Day` | Catégorielle | Moment de la journée où le trajet a lieu | `Morning`, `Afternoon`, `Evening`, `Night` |
| `Day_of_Week` | Catégorielle | Type de jour du trajet | `Weekday` (semaine), `Weekend` (week-end) |
| `Passenger_Count` | Numérique (discrète) | Nombre de passagers à bord | 1 – 4 |
| `Traffic_Conditions` | Catégorielle (ordinale) | Niveau de congestion du trafic | `Low`, `Medium`, `High` |
| `Weather` | Catégorielle | Conditions météorologiques pendant le trajet | `Clear` (dégagé), `Rain` (pluie), `Snow` (neige) |
| `Base_Fare` | Numérique (continue) | Tarif de prise en charge, fixe au démarrage de la course | 2,01 – 5,00 |
| `Per_Km_Rate` | Numérique (continue) | Tarif appliqué par kilomètre parcouru | 0,50 – 2,00 |
| `Per_Minute_Rate` | Numérique (continue) | Tarif appliqué par minute de trajet | 0,10 – 0,50 |
| `Trip_Duration_Minutes` | Numérique (continue) | Durée totale du trajet, en minutes | 5,01 – 119,84 min (médiane ≈ 61,9) |
| **`Trip_Price`** | Numérique (continue) | **Variable cible** : prix total payé pour la course | 6,13 – 332,04 (médiane ≈ 50,1) |


In [ ]:
# Import les données
df = pd.read_csv("/content/drive/MyDrive/machine learning/car_dataset.csv")
df.head()

In [ ]:
# Dimension des données
df.shape

In [ ]:
# Afficher les infos des données
df.info()

### <font color = 'bisque'>Exploration et préparation des données

In [ ]:
# sélection les variables quantitatives
quant_data = df.select_dtypes(exclude="object")

In [ ]:
# décrire les variables quantitatives
quant_data.describe().T

In [ ]:
# pairplot des variables quantitatives
sns.pairplot(quant_data)
plt.show()

In [ ]:
# Fonction permettant de visualiser les boxplots
def boxplot_func(col):
    fig = px.box(df,x = col)
    fig.show()

In [ ]:
# Appliquer la fonctions boxplot_func
for col in quant_data.columns:
  boxplot_func(col)

In [ ]:
# Afficher la matrice de corrélation
quant_data.corr()

In [ ]:
# Visualisaton le matrice de corrélation
plt.figure(figsize = (20, 10))
sns.heatmap(quant_data.corr(), annot = True)
plt.show()

In [ ]:
# sélection les variables catégorielles
cat_data = df.select_dtypes(exclude="number")

In [ ]:
# Faire un describre
cat_data.describe().T

In [ ]:
# Fonction de visualisaion du counplot
def countplot_func(col):
    val = cat_data[col].value_counts()
    fig = px.bar(x = val.index, y = val.values,color = val.index.astype(str), text_auto = True)
    fig.show()

In [ ]:
# Appliquer la fonction countplot_func
for col in cat_data.columns:
  boxplot_func(col)

In [ ]:
# Imputer les valeurs manquantes des variables catégorielle en utilisant le mode
for col in cat_data.columns:
  df.fillna({col: df[col].mode()[0]}, inplace = True)

In [ ]:
df.isna().sum()

In [ ]:
# Enregistrer les valeurs uniques des variables catégorielles dans un objet joblib
uniques = []
for col in cat_data.columns:
  uniques.append(np.unique(df[col]).tolist())
# sauvegarde
import joblib as jb
jb.dump(uniques, 'uniques.joblib')

In [ ]:
# Encoder les variables catégorielles
from sklearn.preprocessing import LabelEncoder
# Créer différents encodeurs pour les variables catégorielles
encoders = []
for i in range(len(cat_data.columns)):
  encoder = LabelEncoder()
  encoder.fit(df[cat_data.columns[i]]) # le modèle prend le temps de reconnaitre les classes
  df[cat_data.columns[i]] = encoder.transform(df[cat_data.columns[i]]) # encoder les valeurs
  # rajouter l'encoder sur la liste encoders
  encoders.append(encoder)

In [ ]:
df.head()

In [ ]:
import joblib as jb
# sauvegarde les encodeurs
jb.dump(encoders, 'encoders.joblib')

In [ ]:
# Recherche du nombre optimal de voisins du KNNImputer avec GridSearchCV
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, KFold

# données dont les valeurs aberrantes (méthode IQR) sont remplacées par des NaN (variables explicatives)
df_nan = df.copy()
for col in quant_data.columns[:-1]:
  Q1 = df_nan[col].quantile(0.25)
  Q3 = df_nan[col].quantile(0.75)
  IQR = Q3 - Q1
  df_nan.loc[(df_nan[col] < Q1 - 1.5 * IQR) | (df_nan[col] > Q3 + 1.5 * IQR), col] = np.nan
# on garde les lignes dont la variable cible est connue
df_nan = df_nan.dropna(subset=[df.columns[-1]])
x_nan = df_nan.iloc[:, :-1]
y_nan = df_nan.iloc[:, -1]

# pipeline : KNNImputer puis modèle
pipe_knn = Pipeline([('imputer', KNNImputer()), ('model', RandomForestRegressor(n_estimators=50, random_state=42))])
# dictionnaire params : nombre de voisins à tester
params_knn = {'imputer__n_neighbors': [1, 3, 5, 7, 9, 11, 15]}
# validation croisée
cv_knn = KFold(n_splits=5, shuffle=True, random_state=42)
# instantier le modèle gridsearch
grid_knn = GridSearchCV(pipe_knn, params_knn, cv=cv_knn)
# lancer la recherche
grid_knn.fit(x_nan, y_nan)
# nombre optimal de voisins
best_k = grid_knn.best_params_['imputer__n_neighbors']
print('Nombre optimal de voisins :', best_k)

In [ ]:
from sklearn.impute import KNNImputer
# Imputer les valeurs manquantes des variables quantitatives
imputer = KNNImputer(n_neighbors=best_k)
df[:] = imputer.fit_transform(df)

In [ ]:
# fonction d'imputation des valeurs aberrantes
def impute_func(df, col, n_neighbors=5):
    # 2. Détection des outliers (Méthode IQR)
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    min_value = Q1 - 1.5 * IQR
    max_value = Q3 + 1.5 * IQR

    # Les valeurs hors bornes deviennent des valeurs manquantes pour le KNNImputer
    df.loc[(df[col] < min_value) | (df[col] > max_value), col] = np.nan # NaN
    # n_neighbors : nombre de voisins pour calculer la moyenne de remplacement
    imputer = KNNImputer(n_neighbors=n_neighbors)
    df[:] = imputer.fit_transform(df)

In [ ]:
# Application du traitement
for col in quant_data.columns :
  impute_func(df,col,n_neighbors=best_k)

In [ ]:
# réafficher les boxplots
for col in quant_data.columns:
  boxplot_func(col)

In [ ]:
# Fractionner les données
x = df.iloc[:,:-1].values
y = df.iloc[:, -1].values

In [ ]:
# importer RobustScaler
from sklearn.preprocessing import RobustScaler
# Normaliser x
scaler = RobustScaler()
scaler.fit(x)
x = scaler.transform(x)

In [ ]:
# sauvegarde du scaler
jb.dump(scaler, 'scaler.joblib')

In [ ]:
from sklearn.model_selection import train_test_split
# Splitter les données en train, val et test
x_train, x_vt,y_train,y_vt = train_test_split(x,y , test_size = 0.2, random_state = 42)
# Splitter les données en val et test
x_val, x_test, y_val, y_test = train_test_split(x_vt, y_vt, test_size = 0.5, random_state = 42)

In [ ]:
x_train[:5]

### <font color = 'bisque'>Multiple Linear Regression

In [ ]:
# Rechercher les paramètres optimaux de l'algorithme
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.linear_model import LinearRegression

# Validation croisée et figé
cv = KFold(n_splits=5, shuffle=True, random_state=42)
# dictionnaire params
params_lr = {
    'fit_intercept': [True, False],
    'positive': [True, False]
 }
# Initialisation du modèle
lr = LinearRegression()
# Initialisation du GridSearchCV
grid_lr = GridSearchCV(lr, params_lr, cv = cv)
# Lancer la recherche
grid_lr.fit(x_train, y_train)

In [ ]:
# Récupérer le modèle optimal en partant de gridsearch
lr_model = grid_lr.best_estimator_

#### <font color = 'teal'> Evaluation du Linear Regression<font>

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error,root_mean_squared_error
# Calculer le R2 score et Mean Squarred Error
# from math import *
def evaluation(model):
  y_pred = model.predict(x_val)
  r2 = r2_score(y_val, y_pred)
  mse = mean_squared_error(y_val, y_pred)
  mae = mean_absolute_error(y_val, y_pred)
  rmse = root_mean_squared_error(y_val, y_pred)
  return r2, mse, mae, rmse

In [ ]:
#Evaluation
r2_lr, mse_lr, mae_lr, rmse_lr = evaluation(lr_model)
print(f"R2 : {r2_lr} -- MSE : {mse_lr} -- RMSE : {rmse_lr} -- MAE : {mae_lr}")

### <font color = 'bisque'>Ridge Regression

In [ ]:
# Rechercher les paramètres optimaux de l'algorithme
from sklearn.linear_model import Ridge
# dictionnaire params
params_rr = {
    'fit_intercept': [True, False],
    'positive': [True, False],
    'alpha':np.logspace(-8,8, 100) #
 }
# Initialisation du modèle
rr = Ridge()
# Initialisation du GridSearchCV
grid_rr = GridSearchCV(rr, params_rr, cv = cv)
# Lancer la recherche
grid_rr.fit(x_train, y_train)

In [ ]:
# Récupérer le modèle optimal en partant de gridsearch
rr_model = grid_rr.best_estimator_

#### <font color = 'teal'> Evaluation du Ridge<font>

In [ ]:
# Evaluation le modèle
r2_rr, mse_rr, mae_rr, rmse_rr = evaluation(rr_model)
print(f"R2 : {r2_rr} -- MSE : {mse_rr} -- RMSE : {rmse_rr} -- MAE : {mae_rr}")

### <font color = 'bisque'>Decision Tree Regressor<font>

In [ ]:
# Recherche des paramètres optimaux du decision tree
from sklearn.tree import DecisionTreeRegressor
# Dictionaire des params
params_dt = {
    'max_depth':np.arange(1,30),
    'criterion':['squared_error', 'absolute_error']
}
# Initialisation du modèle
dt = DecisionTreeRegressor(random_state = 42)
# Initialisation du GridSearchCV
grid_dt = GridSearchCV(dt, params_dt, cv = cv)
# Lancer la recherche
grid_dt.fit(x_train, y_train)

In [ ]:
# Récupérer le modèle optimal en partant de gridsearch
dt_model = grid_dt.best_estimator_

#### <font color = 'teal'> Evaluation du Decision Tree Regressor<font>

In [ ]:
# Evaluation le modèle
r2_dt, mse_dt, mae_dt, rmse_dt = evaluation(dt_model)
print(f"R2 : {r2_dt} -- MSE : {mse_dt} -- RMSE : {rmse_dt} -- MAE : {mae_dt}")

### <font color = 'bisque'>Random Forest Regressor<font>

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Dictionnaires des paramètres
params_rf = {
    'n_estimators':np.arange(10,100,10),
    'max_depth':np.arange(1,15)
}
# Initialisation du modèle
rf =  RandomForestRegressor()
# Objet grid
grid_rf = GridSearchCV(rf, params_rf, cv=cv)
# Lancer la recherche
grid_rf.fit(x_train, y_train)

In [ ]:
# Récupérer le modèle optimal en partant de gridsearch
rf_model = grid_rf.best_estimator_

#### <font color = 'teal'> Evaluation du Random Forest<font>

In [ ]:
# Evaluation le modèle
r2_rf, mse_rf, mae_rf, rmse_rf = evaluation(rf_model)
print(f"R2 : {r2_rf} -- MSE : {mse_rf} -- RMSE : {rmse_rf} -- MAE : {mae_rf}")

In [ ]:
# comparaison des métriques
list_names = ['LR', 'RR', 'DT', 'RF']
list_r2 = [r2_lr, r2_rr, r2_dt, r2_rf]
list_mse = [mse_lr, mse_rr, mse_dt, mse_rf]
list_rmse = [rmse_lr, rmse_rr, rmse_dt, rmse_rf]
list_mae = [mae_lr, mae_rr, mae_dt, mae_rf]
list_final = [list_r2, list_mse, list_rmse, list_mae]

for i in range(4):
  fig = px.bar(y = list_names, x = list_final[i], color = list_names, text_auto = True)
  fig.show()

In [ ]:
# télécharger le modèle le plus performant
jb.dump(rf_model, 'rf_model.joblib')

### <font color = 'bisque'>Tester le modèle le plus performant<font>

In [ ]:
#  echantillon des données test
x_test_10 = x_test[:10]
y_test_10 = y_test[:10]
# Prédictions
y_pred_10 = rf_model.predict(x_test_10)
# Parcourir les prédictions
for i in range(10):
  print(f"Prédiction {i + 1} : {y_pred_10[i]:.2f} --Réel :{y_test_10[i]:.2f}")